# Merge Data Sets

In [165]:
import pandas as pd
import numpy as np
import glob
import os
import re, math, unicodedata
from pathlib import Path

In [166]:
# Define file paths
PATH_SELLER_A = 'data/seller-a/'
PATH_SELLER_B = 'data/seller-b/nish_catalog_run.csv'
PATH_SELLER_C = 'data/seller-c/jngems_custom_ignore.csv'
PATH_SELLER_D = 'data/seller-d/ratnapura.csv'

# Define currency conversion rate
USD_TO_LKR = 330.0

# Define regex pattern for identifying pairs or sets
PAIR_PAT = re.compile(r"\b(pair|set|lot|parcel|pcs|pieces|two\s*stones?|matching|couple)\b", re.I)

# Initialize list to hold dataframes
frames = []

## Merge All Seller A Data set into one master

In [167]:
all_files = glob.glob(PATH_SELLER_A + '*.csv')

dataframes = []
for file in all_files:
    df = pd.read_csv(file)
    dataframes.append(df)

merged_df = pd.concat(dataframes, ignore_index=True)
# Save the merged DataFrame to a new CSV file
merged_df.to_csv(PATH_SELLER_A + 'master-a.csv', index=False)
print("Merged Seller A DataFrame. Shape:", merged_df.shape)

PATH_SELLER_A = 'data/seller-a/master-a.csv'
df_a = pd.read_csv(PATH_SELLER_A)
df_a.head()


Merged Seller A DataFrame. Shape: (670, 14)


,name,price_text,price_value,currency,url,gem_type,weight_carat,clarity,size_mm,colour,shape_cut,treatment,scraped_at,raw_description_text
0,1.16ct Natural Unheated Pink Sapphire,"Rs 713,500.00",713500.0,LKR,https://wijayagems.com/products/1-16ct-natural...,Natural Unheated Pink Sapphire,1.16,Eye Clean Size,NaN,Vivid Pink Shape,Oval Treatment,Unheated Certificate,2025-09-01T12:54:52.564270+00:00,Gem type: Natural Unheated Pink Sapphire\nWeig...
1,1.30ct Natural Unheated Pink Sapphire,"Rs 590,900.00",590900.0,LKR,https://wijayagems.com/products/1-30ct-natural...,Natural Unheated Pink Sapphire,1.30,Eye Clean Size,NaN,Vivid Pink Shape,Oval Treatment,Unheated Certificate,2025-09-01T12:55:01.449590+00:00,Gem type: Natural Unheated Pink Sapphire\nWeig...
2,0.77ct Natural Unheated Pink Sapphire,"Rs 356,800.00",356800.0,LKR,https://wijayagems.com/products/0-77ct-natural...,Natural Pink Sapphire,0.77,Eye Clean Size,NaN,Vivid Pink Shape,Cushion Treatment,Unheated,2025-09-01T12:55:08.550022+00:00,Gem type: Natural Pink Sapphire\nWeight: 0.77 ...
3,0.89ct Natural Unheated Pink Sapphire,"Rs 412,500.00",412500.0,LKR,https://wijayagems.com/products/0-89ct-natural...,Natural Pink Sapphire,0.89,Eye Clean Size,NaN,Vivid Pink Shape,Cushion Treatment,Unheated,2025-09-01T12:55:16.102790+00:00,Gem type: Natural Pink Sapphire\nWeight: 0.89 ...
4,0.93ct Natural Unheated Pink Sapphire Pair,"Rs 245,300.00",245300.0,LKR,https://wijayagems.com/products/0-93ct-natural...,Natural Unheated Pink Sapphire,0.93,NaN,5 mm,Reddish Pink Size,Baguete SKU,NaN,2025-09-01T12:55:24.452816+00:00,Gem type: Natural Unheated Pink Sapphire\nWeig...


### Helper Methods
Helper functions for data normalization and cleaning

# Normalize text by stripping whitespace, normalizing unicode, and removing extra spaces
def _norm_text(x):
    if pd.isna(x): return x
    s = str(x)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0"," ").replace("\u200B","")
    s = re.sub(r"\s+"," ", s).strip()
    return s

# Convert to float, stripping non-numeric characters
def _to_float(x):
    if pd.isna(x) or x == "": return np.nan
    if isinstance(x,(int,float)): return float(x)
    s = re.sub(r"[^\d.\-eE]", "", str(x).strip())
    try: return float(s)
    except: return np.nan
    
# Parse dimensions in mm from a string
def parse_dimensions_mm(s):
    """'10.1 x 8.4 x 6.8 mm' -> (10.1, 8.4, 6.8)"""
    if pd.isna(s): return np.nan, np.nan, np.nan
    s = _norm_text(s).lower().replace("mm","")
    parts = re.split(r"[x×]", s)
    nums = []
    for p in parts:
        v = _to_float(p)
        if not (pd.isna(v) or np.isnan(v)): nums.append(v)
    while len(nums) < 3: nums.append(np.nan)
    return nums[0], nums[1], nums[2]

# Check if a name indicates a pair or set of items
def is_pair_or_set(name):
    if pd.isna(name): return False
    return bool(PAIR_PAT.search(str(name)))

# Normalize shape names
def norm_shape(s):
    s = _norm_text(s)
    if pd.isna(s): return np.nan
    low = s.lower()
    low = (low.replace("round brilliant","round")
               .replace("pear shape","pear")
               .replace("cushion shape","cushion")
               .replace("octagon","emerald cut")
               .replace("radiant cut","radiant"))
    return low.title()

# Normalize colour names, converting "Color" to "Colour"
def norm_colour(c):
    c = _norm_text(c)
    if pd.isna(c): return np.nan
    return c.title().replace("Color","Colour")

# Normalize clarity grades to standard abbreviations
def norm_clarity(c):
    c = _norm_text(c)
    if pd.isna(c): return np.nan
    up = c.upper()
    up = (up.replace("EYE CLEAN","EC")
            .replace("LOUPE CLEAN","LC")
            .replace("VERY VERY SLIGHTLY INCLUDED","VVS")
            .replace("VERY SLIGHTLY INCLUDED","VS")
            .replace("SLIGHTLY INCLUDED","SI")
            .replace("INCLUDED","I"))
    m = re.search(r"\b(LC|VVS|VS|SI|I|EC)\b", up)
    return m.group(1) if m else c

# Normalize treatment descriptions to standard categories
def norm_treatment(t):
    t = _norm_text(t)
    if pd.isna(t): return np.nan
    low = re.sub(r"(certificate.*|videos.*|upon request.*)$","", t.lower()).strip()
    if "unheated" in low or "no heat" in low or "none" in low: return "Unheated"
    if "heated" in low or "heat" in low: return "Heated"
    if "beryllium" in low: return "Be Diffusion"
    if "diffusion" in low: return "Diffusion"
    if "oiling" in low or "oiled" in low: return "Oiled"
    if "filled" in low: return "Fracture Filled"
    return _norm_text(low.title())

# Flag if a certificate is mentioned in the text
def cert_flag(x):
    x = _norm_text(x)
    if pd.isna(x): return 0
    s = x.lower()
    return 1 if any(k in s for k in ["cert","certificate","gia","igi","ngja","gic"]) else 0

# Return a Series from df[col] if it exists, else a filler Series
def series_or(df, col, fill=np.nan):
    """Return df[col] if exists, else a filler Series (so .map() won't crash)."""
    return df[col] if col in df.columns else pd.Series([fill]*len(df), index=df.index)

# Bucket price into nearest k (default 1000)
def price_bucket(x, k=1000):
    try: return int(round(float(x)/k)*k)
    except: return np.nan

In [168]:
def _norm_text(x):
    if pd.isna(x): return x
    s = str(x)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0"," ").replace("\u200B","")
    s = re.sub(r"\s+"," ", s).strip()
    return s

def _to_float(x):
    if pd.isna(x) or x == "": return np.nan
    if isinstance(x,(int,float)): return float(x)
    s = re.sub(r"[^\d.\-eE]", "", str(x).strip())
    try: return float(s)
    except: return np.nan

def parse_dimensions_mm(s):
    # '10.1 x 8.4 x 6.8 mm' -> (10.1, 8.4, 6.8)
    if pd.isna(s): return np.nan, np.nan, np.nan
    s = _norm_text(s).lower().replace("mm","")
    parts = re.split(r"[x×]", s)
    nums = []
    for p in parts:
        v = _to_float(p)
        if not (pd.isna(v) or np.isnan(v)): nums.append(v)
    while len(nums) < 3: nums.append(np.nan)
    return nums[0], nums[1], nums[2]

def fmt_dims(l, w, h):
    vals = [l, w, h]
    ok = [v for v in vals if not (pd.isna(v) or np.isnan(v))]
    if not ok: return ""
    txt = " x ".join([f"{v:.2f}".rstrip("0").rstrip(".") for v in ok]) + " mm"
    return txt

def series_or(df, col, fill=np.nan):
    return df[col] if col in df.columns else pd.Series([fill]*len(df), index=df.index)

def cert_yesno_from_row(row, scan_cols):
    # “Yes” if any column hints a cert; else “No”
    KEYS = ("cert","certificate","gia","igi","gic","ngja","ngtc")
    for c in scan_cols:
        if c in row.index and pd.notna(row[c]):
            s = str(row[c]).lower()
            if any(k in s for k in KEYS): 
                return "Yes"
    # explicit numeric flag also counts
    for c in ("certificate_flag",):
        if c in row.index and pd.notna(row[c]):
            try:
                if int(row[c]) == 1: 
                    return "Yes"
            except:
                pass
    return "No"

def date_from_any(df, candidates=("scraped_at","date","Date","created_at","updated_at")):
    for c in candidates:
        if c in df.columns:
            dt = pd.to_datetime(df[c], errors="coerce")
            if dt.notna().any():
                return dt.dt.date.astype("string")
    return pd.Series([""]*len(df), index=df.index, dtype="string")

def split_gemtype_variety(text):
    """
    Returns (Gem Type, Variety).
    If text contains 'sapphire' -> Gem Type='Sapphire', Variety=Title(text) (ensure ends with 'Sapphire').
    Else Gem Type and Variety both = Title(text) (e.g., 'Ruby', 'Emerald').
    """
    if pd.isna(text) or str(text).strip()=="":
        return "", ""
    t = _norm_text(str(text)).replace("-", " ").strip().lower()
    vt = t.title()
    if "sapphire" in t:
        # ensure 'Sapphire' suffix for colours like 'Padparadscha'
        if "sapphire" not in vt.lower():
            vt = vt + " Sapphire"
        return "Sapphire", vt
    # common single-type gems
    for base in ["ruby","emerald","spinel","alexandrite","tourmaline","topaz","aquamarine","garnet"]:
        if base in t:
            return base.title(), vt
    # fallback
    return vt, vt

def build_sellers_schema(df, source_hint="", colmap=None, scan_cert_cols=None, dim_cols=None):
    """
    Returns a DF with exactly the sellers columns + Date:
    Gem Type, Variety, Carat Weight (ct), Cut/Shape, Colour, Clarity, Treatments,
    Asking Price (LKR), Final Selling Price (if available), Dimensions (LxWxH, mm),
    Certification Status, Date
    """
    colmap = colmap or {}
    scan_cert_cols = scan_cert_cols or []
    dim_cols = dim_cols or {}

    # ---- Source-specific raw columns ----
    name = series_or(df, colmap.get("name","name"), "")
    category = series_or(df, colmap.get("category","category"), "")
    gem_type_raw = series_or(df, colmap.get("gem_type","gem_type"), "")
    # prefer explicit gem_type/category, else fall back to name
    base_txt = gem_type_raw.where(gem_type_raw.astype(str).str.len()>0, category)
    base_txt = base_txt.where(base_txt.astype(str).str.len()>0, name)

    GT, V = [], []
    for s in base_txt.fillna(""):
        g, v = split_gemtype_variety(s)
        GT.append(g); V.append(v)

    # weights
    wt = series_or(df, colmap.get("weight","weight_carat"), np.nan).map(_to_float)

    # shape/colour/clarity/treatments
    shape = series_or(df, colmap.get("shape","shape"), "")
    colour = series_or(df, colmap.get("colour","colour"), "")
    clarity = series_or(df, colmap.get("clarity","clarity"), "")
    treat = series_or(df, colmap.get("treatment","treatment"), "")

    # price LKR
    price_col = colmap.get("price_lkr")
    price = series_or(df, price_col, np.nan).map(_to_float)

    # final selling price (usually absent)
    final_price = series_or(df, colmap.get("final_price",""), "").map(_to_float)

    # dimensions
    L = series_or(df, dim_cols.get("L","length_mm"), np.nan).map(_to_float)
    W = series_or(df, dim_cols.get("W","width_mm"), np.nan).map(_to_float)
    H = series_or(df, dim_cols.get("H","depth_mm"), np.nan).map(_to_float)
    dim_str = [fmt_dims(l,w,h) for l,w,h in zip(L,W,H)]

    # if no L/W, try parsing size text
    if (pd.isna(L).all() and pd.isna(W).all()):
        size_txt = series_or(df, dim_cols.get("size","size_mm"), "")
        dims = size_txt.apply(parse_dimensions_mm)
        dim_str = [fmt_dims(a,b,c) for (a,b,c) in dims]

    # certificate status
    cert = [cert_yesno_from_row(df.loc[i], scan_cert_cols) for i in df.index]

    # date
    date_ser = date_from_any(df, candidates=colmap.get("date_candidates", ("scraped_at","date","Date","created_at","updated_at")))

    # build sellers schema
    out = pd.DataFrame({
        "Gem Type": GT,
        "Variety": V,
        "Carat Weight (ct)": wt.round(2),
        "Cut/Shape": shape.map(lambda s: _norm_text(s).title() if isinstance(s,str) else ""),
        "Colour": colour.map(lambda s: _norm_text(s).title() if isinstance(s,str) else ""),
        "Clarity": clarity.map(lambda s: _norm_text(s).upper() if isinstance(s,str) else ""),
        "Treatments": treat.map(lambda s: _norm_text(s).title() if isinstance(s,str) else ""),
        "Asking Price (LKR)": price,
        "Final Selling Price (if available)": final_price,   # keep blank if unknown
        "Dimensions (LxWxH, mm)": dim_str,
        "Certification Status": cert,  # Yes/No
        "Date": date_ser.fillna("").astype(str),
    })

    # remove pair/set rows using name/category if available
    name_for_filter = name.where(name.astype(str).str.len()>0, category)
    mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)
    out = out[~mask_pair].copy()

    # tidy blanks
    out["Gem Type"] = out["Gem Type"].fillna("")
    out["Variety"] = out["Variety"].fillna("")
    out["Certification Status"] = out["Certification Status"].replace({"": "No"})

    return out

# Read & normalize each source into the SAME schema

## Seller - C

In [169]:
if Path(PATH_SELLER_C).exists():
    jn = pd.read_csv(PATH_SELLER_C)
    frames.append(build_sellers_schema(
        jn,
        colmap={
            "name": "title",
            "category": "category_slug",
            "gem_type": "category_slug",
            "weight": "weight_carat",
            "shape": "shape_cut",
            "colour": "colour",
            "clarity": "clarity",
            "treatment": "heated",
            "price_lkr": "price_current_value",
            "date_candidates": ("scraped_at",),
        },
        scan_cert_cols=["certificate","product_description"]
    ))

/var/folders/x0/kj7rxcb54sb06z9f6ylxnt000000gn/T/ipykernel_9841/485891275.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)


## Seller - B

In [170]:
if Path(PATH_SELLER_B).exists():
    ng = pd.read_csv(PATH_SELLER_B)
    # convert price to LKR first
    if "price_value" in ng.columns:
        ng["price_lkr"] = ng["price_value"].map(_to_float) * USD_TO_LKR
    frames.append(build_sellers_schema(
        ng,
        colmap={
            "name": "name",
            "category": "category",
            "gem_type": "category",
            "weight": "weight_cts",
            "shape": "shape",
            "colour": "colour",
            "clarity": "clarity",
            "treatment": "treatment",
            "price_lkr": "price_lkr",
            "date_candidates": ("scraped_at",),
        },
        scan_cert_cols=["certificate","description"],
        dim_cols={"size": "dimensions"}  # Nish has free-text dimensions
    ))

/var/folders/x0/kj7rxcb54sb06z9f6ylxnt000000gn/T/ipykernel_9841/485891275.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)


## Seller - A

In [171]:
if Path(PATH_SELLER_A).exists():
    wz = pd.read_csv(PATH_SELLER_A)
    frames.append(build_sellers_schema(
        wz,
        colmap={
            "name": "name",
            "category": "gem_type",
            "gem_type": "gem_type",
            "weight": "weight_carat",
            "shape": "shape_cut",
            "colour": "colour",
            "clarity": "clarity",
            "treatment": "treatment",
            "price_lkr": "price_value",
            "date_candidates": ("scraped_at",),
        },
        scan_cert_cols=["price_text","raw_description_text"],
        dim_cols={"size": "size_mm"}
    ))


/var/folders/x0/kj7rxcb54sb06z9f6ylxnt000000gn/T/ipykernel_9841/485891275.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)


In [172]:
frames

[      Gem Type             Variety  Carat Weight (ct)  \
 0     Sapphire     Ceylon Sapphire               5.88   
 1     Sapphire     Ceylon Sapphire               0.86   
 2     Sapphire     Ceylon Sapphire               0.77   
 3     Sapphire     Ceylon Sapphire               1.24   
 4     Sapphire     Ceylon Sapphire               1.77   
 ..         ...                 ...                ...   
 257   Ametrine            Ametrine                NaN   
 258   Ametrine            Ametrine                NaN   
 259   Sapphire  Bi Color Sapphires               2.15   
 260   Sapphire  Bi Color Sapphires               4.54   
 263  Moonstone           Moonstone               1.11   
 
                     Cut/Shape  \
 0             Cushion Mix Cut   
 1                  Oval ,Step   
 2                   Oval Step   
 3                   Oval Step   
 4                   Oval Step   
 ..                        ...   
 257              Octagon Step   
 258              Octagon Step

## Seller - D

In [173]:
if Path(PATH_SELLER_D).exists():
    sd = pd.read_csv(PATH_SELLER_D)
    # Try to align columns if already in Sellers format
    # If your sheet already has the exact names, just rename/keep
    possible = sd.copy()
    # unify headings if slightly different
    rename_map = {
        "GemType": "Gem Type",
        "Gem type": "Gem Type",
        "Variety ": "Variety",
        "Carat Weight": "Carat Weight (ct)",
        "Cut/Shape ": "Cut/Shape",
        "Color": "Colour",
        "Treatment": "Treatments",
        "Asking Price (LKR) ": "Asking Price (LKR)",
        "Final Selling Price": "Final Selling Price (if available)",
        "Certification": "Certification Status",
        "Dimensions": "Dimensions (LxWxH, mm)",
        "Date ": "Date",
    }
    possible.rename(columns=rename_map, inplace=True)
    needed = ["Gem Type","Variety","Carat Weight (ct)","Cut/Shape","Colour","Clarity","Treatments",
              "Asking Price (LKR)","Final Selling Price (if available)","Dimensions (LxWxH, mm)","Certification Status","Date"]
    if set(needed).issubset(set(possible.columns)):
        # just take & coerce types
        chunk = possible[needed].copy()
        # normalize types a bit
        chunk["Asking Price (LKR)"] = chunk["Asking Price (LKR)"].map(_to_float)
        chunk["Final Selling Price (if available)"] = chunk["Final Selling Price (if available)"].map(_to_float)
        frames.append(chunk)
    else:
        # Fallback: build from raw with flexible scan of cert & dimension columns
        frames.append(build_sellers_schema(
            sd,
            colmap={
                "name": "name",
                "category": "Gem Type",
                "gem_type": "Gem Type",
                "weight": "Carat Weight (ct)",
                "shape": "Cut/Shape",
                "colour": "Colour",
                "clarity": "Clarity",
                "treatment": "Treatments",
                "price_lkr": "Asking Price (LKR)",
                "final_price": "Final Selling Price (if available)",
                "date_candidates": ("Date","scraped_at"),
            },
            scan_cert_cols=list(sd.columns),
            dim_cols={"size": "Dimensions (LxWxH, mm)"}
        ))

/var/folders/x0/kj7rxcb54sb06z9f6ylxnt000000gn/T/ipykernel_9841/485891275.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)


In [174]:
if not frames:
    raise RuntimeError("No input files found. Check the PATH_* variables at the top.")
# Combine all frames
master = pd.concat(frames, ignore_index=True)

# basic final tidy
master["Certification Status"] = master["Certification Status"].replace({"": "No"}).fillna("No")
master["Asking Price (LKR)"] = master["Asking Price (LKR)"].map(_to_float)
master["Final Selling Price (if available)"] = master["Final Selling Price (if available)"].map(_to_float)
master["Date"] = master["Date"].fillna("").astype(str)

# keep EXACT column order
cols = ["Gem Type","Variety","Carat Weight (ct)","Cut/Shape","Colour","Clarity","Treatments",
        "Asking Price (LKR)","Final Selling Price (if available)","Dimensions (LxWxH, mm)","Certification Status","Date"]
master = master[cols].reset_index(drop=True)

# remove totally empty rows (no Gem Type & no Dimensions)
mask_empty = (master["Gem Type"].astype(str).str.strip()=="") & (master["Dimensions (LxWxH, mm)"].astype(str).str.strip()=="")
master = master[~mask_empty].copy()

OUT = Path("data/master_research_schema.csv")
master.to_csv(OUT, index=False)

print(f"Saved -> {OUT.resolve()}")
print("Rows:", len(master))
print(master.head(10))

Saved -> /Users/helithasri/Personal/Gem Price Prediction System Ratnapura Srilanka/notebooks/data/master_research_schema.csv
Rows: 946
   Gem Type          Variety  Carat Weight (ct)            Cut/Shape  \
0  Sapphire  Ceylon Sapphire               5.88      Cushion Mix Cut   
1  Sapphire  Ceylon Sapphire               0.86           Oval ,Step   
2  Sapphire  Ceylon Sapphire               0.77            Oval Step   
3  Sapphire  Ceylon Sapphire               1.24            Oval Step   
4  Sapphire  Ceylon Sapphire               1.77            Oval Step   
5  Sapphire  Ceylon Sapphire               1.18           Pear ,Step   
6  Sapphire  Ceylon Sapphire               1.23  Asscher Diamond Cut   
7  Sapphire  Ceylon Sapphire               1.55            Oval Step   
8  Sapphire  Ceylon Sapphire               1.88            Oval Step   
9  Sapphire  Ceylon Sapphire               2.24            Oval Step   

           Colour      Clarity Treatments  Asking Price (LKR)  \
0   Gre

In [175]:
master


,Gem Type,Variety,Carat Weight (ct),Cut/Shape,Colour,Clarity,Treatments,Asking Price (LKR),Final Selling Price (if available),"Dimensions (LxWxH, mm)",Certification Status,Date
0,Sapphire,Ceylon Sapphire,5.88,Cushion Mix Cut,Greenish Blue,CLEAN,,1155800.0,NaN,,No,2025-09-02
1,Sapphire,Ceylon Sapphire,0.86,"Oval ,Step",Greenish Blue,CLEAN,,46300.0,NaN,6.9 x 4.9 mm,No,2025-09-02
2,Sapphire,Ceylon Sapphire,0.77,Oval Step,Pinkish Orange,CLEAN,,80200.0,NaN,6.8 x 5.2 mm,No,2025-09-02
3,Sapphire,Ceylon Sapphire,1.24,Oval Step,Vivid,CLEAN,,84800.0,NaN,7.5 x 5.8 mm,No,2025-09-02
4,Sapphire,Ceylon Sapphire,1.77,Oval Step,Ice Blue,CLEAN,,107900.0,NaN,8.7 x 7 mm,No,2025-09-02
...,...,...,...,...,...,...,...,...,...,...,...,...
941,Sapphire,Sapphire,5.00,Oval,Yellow,IF - INTERNALLY FLAWLESS,Heat Treatment,700000.0,700000.0,,No,
942,Sapphire,Sapphire,8.00,Rough,Red,IF - INTERNALLY FLAWLESS,Heat Treatment,10000000.0,9000000.0,,No,
943,Cat'S Eye,Cat'S Eye,2.00,Round,Green,IF - INTERNALLY FLAWLESS,Heat Treatment,100000.0,10000.0,,No,
944,Spinel,Spinel,1.00,Oval,Pink,IF - INTERNALLY FLAWLESS,Natural,200000.0,190000.0,,No,


In [176]:
master.isnull().sum()

Gem Type                                0
Variety                                 0
Carat Weight (ct)                      38
Cut/Shape                               0
Colour                                  0
Clarity                                 0
Treatments                              0
Asking Price (LKR)                      0
Final Selling Price (if available)    929
Dimensions (LxWxH, mm)                  0
Certification Status                    0
Date                                    0
dtype: int64